# 01 - How LangChain Actually Works

LangChain has a reputation for being a confusing pile of abstractions. Most of that
confusion comes from tutorials written for LangChain 0.x, whose central object —
`AgentExecutor` — no longer exists in modern LangChain.

This notebook covers the small number of pieces that actually matter in **LangChain 1.x**.

By the end you will be able to:
1. Say what LangChain gives you that calling a provider SDK directly does not.
2. Use messages, models, prompts, tools and structured output.
3. Read a tool call off a model response and execute it.
4. Recognise deprecated 0.x patterns when you meet them in old tutorials.

> **No API key needed.** Every cell runs against a scripted stand-in model
> (`offline_llm.py`). Swap in a real provider at any point with `real_model()`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))     # so `offline_llm` is importable

from offline_llm import ScriptedChatModel, ai, tool_call
import langchain
print("langchain version:", langchain.__version__)

## 1. What LangChain is for

Strip away the marketing and LangChain does three useful things:

1. **One interface over every provider.** The same code runs against OpenAI, Anthropic,
   Google or a local Ollama model. Swapping provider is a one-line change.
2. **A standard message and tool-call format.** Every provider has its own JSON shape for
   tool calls. LangChain normalises them, so your tool-handling code is written once.
3. **Composable pieces.** Prompts, models, parsers and tools snap together.

That is genuinely useful. What it is **not** is a reason to use LangChain for a single
API call — for that, the provider's own SDK is simpler. LangChain earns its place when you
have tools, multiple steps, or want provider portability.

## 2. Models

`init_chat_model` builds a model from a `"provider:model"` string.

```python
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4.1-mini")
llm = init_chat_model("anthropic:claude-opus-5")
llm = init_chat_model("google_genai:gemini-2.5-flash")
llm = init_chat_model("ollama:llama3.2")          # local, free, no key
```

Everything downstream is identical regardless of which line you picked. That portability
is the single best reason to use LangChain.

For this notebook we use a scripted stand-in so nothing needs a key.

In [ ]:
llm = ScriptedChatModel(responses=[
    ai("Machine learning is a way of writing programs that learn patterns from data "
       "instead of being given explicit rules."),
])

response = llm.invoke("Explain machine learning in one sentence.")
print(type(response).__name__)
print()
print(response.content)

## 3. Messages

A conversation is a list of typed messages. There are four you will use constantly:

| Message | Who wrote it | Purpose |
|---|---|---|
| `SystemMessage` | you | standing instructions, persona, rules |
| `HumanMessage` | the user | the request |
| `AIMessage` | the model | the reply — **and any tool calls** |
| `ToolMessage` | your code | the result of running a tool |

The whole agent loop in notebook 03 is just these four types in a cycle.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

conversation = [
    SystemMessage("You are a concise data science tutor. Never write more than two sentences."),
    HumanMessage("What is overfitting?"),
]

llm = ScriptedChatModel(responses=[
    ai("Overfitting is when a model memorises the training data instead of the pattern. "
       "You spot it when training loss keeps falling while validation loss rises."),
])

reply = llm.invoke(conversation)
conversation.append(reply)

for m in conversation:
    print(f"{type(m).__name__:15} {m.content[:80]}")

> **The model is stateless.** It has no memory between calls — you resend the whole list
> every time. "Memory" in an agent framework means *managing that list for you*. That is
> what LangGraph's checkpointer does in notebook 02.

## 4. Prompt templates

A template is a string with slots, so you write the prompt once and fill it per request.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Answer in at most {max_words} words."),
    ("human", "{question}"),
])

messages = prompt.invoke({
    "role": "machine learning tutor",
    "max_words": 30,
    "question": "What does a learning rate do?",
})

for m in messages.messages:
    print(f"{type(m).__name__:15} {m.content}")

## 5. Chaining with `|`

The pipe operator connects components: the output of one becomes the input of the next.
This is LangChain Expression Language (LCEL).

In [ ]:
from langchain_core.output_parsers import StrOutputParser

llm = ScriptedChatModel(responses=[
    ai("It controls how big a step the optimiser takes on each update."),
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({
    "role": "machine learning tutor",
    "max_words": 30,
    "question": "What does a learning rate do?",
})
print(repr(result))       # a plain string, not an AIMessage - the parser unwrapped it

Read `prompt | llm | StrOutputParser()` as "fill the template, send it to the model,
then pull the text out". Every chain also gets `.stream()` and `.batch()` for free.

**When to stop chaining.** LCEL is a straight line: A then B then C. The moment you need a
**loop** (call a tool, look at the result, maybe call another) or a **branch** (route to a
different path depending on the answer), you have outgrown chains. That is exactly the
point where LangGraph starts — notebook 02.

## 6. Tools

A tool is a Python function the model is allowed to ask for. The `@tool` decorator turns
a function into one.

In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    fake = {"beirut": "28C and sunny", "london": "12C and raining", "cairo": "35C and clear"}
    return fake.get(city.lower(), f"No data for {city}")

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

print("name       :", get_weather.name)
print("description:", get_weather.description)
print("args schema:", get_weather.args)

> **The docstring is not documentation — it is the prompt.** The model decides whether to
> call your tool based on its name, description and argument names. A vague docstring
> produces a tool the model never calls, or calls at the wrong moment. This is the single
> highest-leverage thing you control when an agent misbehaves.

### The model does not run your tool

This trips up everyone once. The model **requests** a call; **your code** executes it and
sends the result back. Nothing runs automatically.

In [ ]:
llm = ScriptedChatModel(responses=[
    ai("", [tool_call("get_weather", city="Beirut")]),
])

model_with_tools = llm.bind_tools([get_weather, multiply])
response = model_with_tools.invoke([HumanMessage("What is the weather in Beirut?")])

print("content    :", repr(response.content))    # empty - the model wants a tool first
print("tool_calls :", response.tool_calls)

In [ ]:
# Your code executes it and returns a ToolMessage.
call = response.tool_calls[0]
result = get_weather.invoke(call["args"])

tool_message = ToolMessage(content=str(result), tool_call_id=call["id"])
print("executed   :", call["name"], call["args"])
print("result     :", result)
print("wrapped as :", type(tool_message).__name__)

That cycle — **model asks → you execute → you send the result back → model answers** — is
the entire idea of an agent. Everything in notebook 03 is that loop, automated.

## 7. Structured output

Free text is hard to program against. `with_structured_output` makes the model fill in a
schema instead, and you get back a validated Python object.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Ticket(BaseModel):
    """A support ticket extracted from a customer message."""
    category: Literal["billing", "technical", "account", "other"] = Field(
        description="Which team should handle this")
    urgency: Literal["low", "medium", "high"] = Field(description="How fast it needs a reply")
    summary: str = Field(description="One-line summary of the problem")

print("Schema the model must fill:")
for name, field in Ticket.model_fields.items():
    print(f"  {name:10} {field.annotation}")

print("""
With a real model:

    structured = llm.with_structured_output(Ticket)
    ticket = structured.invoke("I have been charged twice this month and nobody replied!")
    ticket.category    # 'billing'
    ticket.urgency     # 'high'

You get a validated Ticket object, not a string you have to parse.""")

**Use structured output whenever the result feeds other code.** Classification, extraction,
routing decisions, filling a database row. Parsing free text with regexes is a bug waiting
to happen.

## 8. Recognising deprecated LangChain 0.x code

You will meet a lot of it. `tool-calling-legacy/exercise1.py` in this folder is written in
the old style. Here is the translation table:

| LangChain 0.x (deprecated) | LangChain 1.x |
|---|---|
| `from langchain.agents import AgentExecutor` | `from langchain.agents import create_agent` |
| `create_tool_calling_agent(llm, tools, prompt)` | `create_agent(model, tools, system_prompt=...)` |
| `AgentExecutor(agent=..., tools=...)` | `create_agent(...)` returns a ready graph |
| `ConversationBufferMemory` | a LangGraph **checkpointer** + `thread_id` |
| `LLMChain(llm=..., prompt=...)` | `prompt \| llm` |
| `initialize_agent(...)` | `create_agent(...)` |

```python
# --- OLD (do not write this) ---
agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, memory=ConversationBufferMemory())
executor.invoke({"input": "..."})

# --- NEW ---
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(model, tools, system_prompt="...", checkpointer=InMemorySaver())
agent.invoke({"messages": [{"role": "user", "content": "..."}]},
             config={"configurable": {"thread_id": "user-123"}})
```

The big conceptual change: **an agent is now a graph**, not an executor object. That is why
notebook 02 is about graphs before notebook 03 builds an agent.

---
## Exercises

1. Write a `@tool` called `search_papers(topic: str, year: int)`. Print `.args` and
   `.description`. Now rewrite the docstring to be vague and compare — what would the model
   lose?
2. Build a chain `prompt | llm | StrOutputParser()` that summarises a paragraph in exactly
   three bullet points. Test the template with `.invoke()` alone first.
3. Extend `Ticket` with a `customer_sentiment` field. Which fields need a `description` for
   the model to fill them correctly, and which are obvious from the name?
4. Open `tool-calling-legacy/exercise1.py` and rewrite it using `create_agent`. Use the
   table above. You will remove more lines than you add.
5. **Stretch:** copy `.sample.env` to `.env`, add one real API key, and re-run sections 2-7
   with `real_model()` instead of `ScriptedChatModel`. Nothing else should need to change —
   that is the portability claim, tested.

## Next

Notebook 02 introduces LangGraph, which is what you use once a straight-line chain is not
enough.